In [ ]:
from sklearn.linear_model import LogisticRegression
%load_ext autoreload
%autoreload 2

import sys
import os

chemin_racine = os.path.abspath('..')

if chemin_racine not in sys.path:
    sys.path.append(chemin_racine)

# 4. Mise en place de la nouvelle baseline

Ici, nous allons pouvoir mettre en place une régression logistique qui se basera sur les données qui seront traitées par notre pipeline de pré-traitement orienté objet, il sera alors pertinent de comparer ses performances avec notre première version qui reposait sur un prétraitement plus manuel avec notamment des valeurs codées en dur.

## a. Import des données et prétraitement :

Ici, comme pour notre première baseline, les données seront traitées via le pipeline de base à laquelle nous ajouterons un `StandardScaler` qui va normaliser nos données pour éviter de biaiser le modèle linéaire de par l'hétérogénéité des plages de valeur des variables.

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from src.pipelines import get_cleaner_pipeline

raw_data_dir = "../data/raw/"
raw_train_val_file = 'cs-training.csv'
raw_test_file = 'cs-test.csv'

credit_data = pd.read_csv(raw_data_dir + raw_train_val_file, index_col=0, header=0)

train_credit_data, val_credit_data = train_test_split(credit_data, test_size=0.2, random_state=67, stratify=credit_data['SeriousDlqin2yrs'])

X_train = train_credit_data.drop('SeriousDlqin2yrs', axis=1)
y_train = train_credit_data['SeriousDlqin2yrs']

X_val = val_credit_data.drop('SeriousDlqin2yrs', axis=1)
y_val = val_credit_data['SeriousDlqin2yrs']

standard_scaler = StandardScaler().set_output(transform='pandas')
preprocess_pipeline = get_cleaner_pipeline()

baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocess_pipeline),
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(class_weight='balanced', random_state=67))
])

## b. Prédictions et métriques de performance :

Comme pour la première baseline, nous allons prédire les probabilités de défaut de crédit des données de validation et mesurer la qualité desdites prédictions en mesurant les métriques AUC-ROC et coeff de Gini.

In [ ]:
from src.utils import evaluate_model_accuracy, display_model_accuracy

baseline_pipeline.fit(X_train, y_train)

val_baseline_accuracy= evaluate_model_accuracy(baseline_pipeline, X_val, y_val)

display_model_accuracy(*val_baseline_accuracy)


Notre nouveau modèle baseline basé sur notre prétraitement orienté objet est une réussite. En effet, on a un AUC-ROC à 0.858 et un coefficient de Gini à 0.716. Ces chiffres sont très satisfaisants puisque c'est un gain de +0.10% sur le Gini, ce qui nous donne trois indicateurs :
- Efficacité du nouveau choix pour `DebtRatio` : Scinder la variable initialement hybride `DebtRatio` en deux variables distinctes a permis de donner davantage de clarté pour notre modèle linéaire qui a donc pu estimer les deux facettes de la variable de façon plus juste.
- Supériorité du capping dynamique : Le transformateur `QuantileCapper` s'est avéré plus efficace que nos valeurs codées en dur.
- Garantie d'absence de data leakage : Le score que l'on vient d'obtenir est forcément étanche puisque les médianes, les indicateurs de valeurs manquantes et les seuils de coupure ont été appris strictement sur le jeu d'entraînement, le modèle arrive donc bien à généraliser sur de la donnée inédite.

## d. Analyse des coefficients $\beta_i$ de la régression :

Comme pour notre première régression, nous allons nous intéresser aux coefficientes $\beta_i$ associés à chacune de nos variables. On rappelle qu'un coefficiant élevé indique qu'une augmentation de la variable $X_i$ fait exploser la probabilité de défaut et à l'inverse, un coefficient négatif montre que plus la variable augmente, plus la probabilité de défaut diminue.

Revoir ces coefficients va nous permettre d'effectuer une comparaison avec les coefficients générés par notre première baseline qui se basait sur notre ancienne méthode de prétraitement.

In [ ]:
logreg_model = baseline_pipeline.named_steps['logreg']

cols_apres_pipeline = baseline_pipeline.named_steps['preprocessor'].transform(X_train.head(1)).columns

coeffs_v2 = pd.DataFrame({
    "Variable": cols_apres_pipeline,
    "Coefficient":logreg_model.coef_[0, :]
})

coeffs_v2.sort_values(by="Coefficient", ascending=False, inplace=True)
print(coeffs_v2)

On obtient le tableau suivant :

### Poids des variables (Coefficients $\beta$) et comparaison avec la Baseline v1

| Variable                               | Coefficient v2 ($\beta$) | Ancien Coef v1 | Évolution / Interprétation                                  |
|----------------------------------------|--------------------------|----------------|-------------------------------------------------------------|
| `RevolvingUtilizationOfUnsecuredLines` | **0.7035**               | 0.7238         | Premier facteur de risque (Surendettement court terme)      |
| `NumberOfTimes90DaysLate`              | **0.5036**               | 0.5017         | Défaut de paiement avéré (Indicateur comportemental stable) |
| `NumberOfTime30-59DaysPastDueNotWorse` | **0.4069**               | 0.4085         | Retard léger (Indicateur comportemental stable)             |
| `NumberOfTime60-89DaysPastDueNotWorse` | **0.3215**               | 0.3208         | Retard modéré (Indicateur comportemental stable)            |
| `NumberOfOpenCreditLinesAndLoans`      | **0.1606**               | 0.1800         | Multiplicité des lignes de crédit                           |
| `MonthlyDebtAmount`                    | **0.1435**               | *N/A*          | **Nouvelle variable :** Charge mensuelle absolue en euros   |
| `Has_System_Error_96_98`               | **0.1195**               | 0.1178         | Flag d'erreur système (Risque identifié)                    |
| `NumberRealEstateLoansOrLines`         | **0.0554**               | 0.1043         | Multiplicité des prêts immobiliers (Baisse du coefficient)  |
| `NumberOfDependents`                   | **0.0367**               | 0.0310         | Charge familiale (Impact mineur)                            |
| `missingindicator_MonthlyIncome`       | **-0.0415**              | -0.0396        | Absence de déclaration de revenus                           |
| `TrueDebtRatio`                        | **-0.0569**              | 0.0253         | **Ancien DebtRatio :** Ratio d'endettement pur              |
| `MonthlyIncome`                        | **-0.2253**              | -0.1262        | Solvabilité financière (Mitigateur de risque plus prononcé) |
| `age`                                  | **-0.2869**              | -0.2960        | Maturité financière (Premier réducteur de risque)           |

* **La stabilité des indicateurs de surendettement et de retards :** En comparant ces résultats avec notre premier modèle, on constate une forte similarité sur les variables à plus fort impact. La variable `RevolvingUtilizationOfUnsecuredLines` (0.7035 contre 0.7238 initialement) ainsi que les compteurs de retards `NumberOfTime...` conservent des coefficients très proches. Cette constance montre que ces indicateurs comportementaux restent le principal facteur de risque de défaut, indépendamment des modifications apportées à notre pipeline de prétraitement.


* **L'apport du découplage de la variable `DebtRatio` :** Contrairement à notre première baseline où le ratio d'endettement global affichait un coefficient faible (0.0253), la création de la variable `MonthlyDebtAmount` permet d'isoler la charge financière absolue avec un coefficient plus élevé (+0.1435). Cette évolution est logique : en fournissant au modèle un montant mensuel à rembourser en euros plutôt qu'une variable hybride, l'algorithme linéaire associe plus nettement le volume de dettes au risque de défaut. On note en parallèle que le coefficient de `NumberRealEstateLoansOrLines` diminue de moitié (passant de 0.1043 à 0.0554), ce qui s'explique probablement par le fait que le poids financier de ces prêts immobiliers est désormais directement absorbé par cette nouvelle variable de mensualité.


* **Le comportement du ratio d'endettement pur (`TrueDebtRatio`) :** Une fois séparée des profils sans revenu qui présentaient des valeurs brutes élevées, la variable mesurant le pourcentage d'endettement voit son coefficient passer de +0.0253 dans la version initiale à -0.0569. Ce résultat peut s'interpréter par le fait que, lorsque le revenu mensuel et le montant des mensualités sont déjà intégrés dans la régression linéaire, un taux d'effort proportionnel et maîtrisé reflète davantage un profil inséré dans le système bancaire ayant eu accès au crédit qu'un risque de défaut imminent.


* **Le revenu mensuel comme mitigateur de risque renforcé :** Comme observé dans la première baseline, la variable `MonthlyIncome` contribue à réduire le risque de défaut. On remarque cependant que son coefficient négatif est plus important dans cette seconde version (-0.2253 contre -0.1262 initialement). Cette différence s'explique par l'utilisation du transformateur `QuantileCapper` basé sur l'algorithme `Kneed` : en cappant les valeurs aberrantes au niveau d'un point d'inflexion plutôt qu'à un seuil fixe codé en dur, la distribution standardisée est plus homogène, ce qui permet au modèle d'accorder plus de poids à la solvabilité financière réelle du client.


* **La linéarité de la variable `age` :** De manière similaire à nos observations initiales, l'âge du client reste le principal facteur de réduction du risque de défaut, avec un coefficient quasi identique (-0.2869 contre -0.2960 initialement). Les critiques formulées lors de la première baseline concernant les limites d'une modélisation strictement linéaire du vieillissement restent applicables ici.

## Bonus : Soumission Kaggle

Ici, on va refaire une soumission Kaggle à la compétition [Give Me Some Credit](https://www.kaggle.com/competitions/GiveMeSomeCredit) pour pouvoir à la fois réévaluer les performances de notre baseline sur les données de test Kaggle, les comparer à nos résultats précédents et toujours dans l'optique de garder un historique des modèles de l'étude.

In [ ]:
from src.kaggle_submissions import generate_kaggle_submission

test_credit_data = pd.read_csv(raw_data_dir + raw_test_file, index_col=0, header=0)
ids = test_credit_data.index
X_test = test_credit_data.drop(labels = ["SeriousDlqin2yrs"], axis = 1)

generate_kaggle_submission(baseline_pipeline,X_test, ids, "submission_baseline_v2.csv")


<br>
<p align="center">
  <img src="../images/kaggle_baseline_v2_score.png" alt="Score Kaggle Baseline LogReg" width="800"/>
</p>
<br>

La soumission de nos prédictions sur la plateforme Kaggle nous donne un score officiel d'AUC-ROC de **0.85744** sur le jeu de test privé et de **0.85192** sur le jeu de test public.

Afin de mesurer précisément l'apport de notre nouvelle architecture orientée objet et de notre remaniement du feature engineering, nous pouvons croiser ces résultats avec ceux de notre première baseline :

### Comparaison des performances officielles (Kaggle)

| Métrique / Jeu de test                           | Baseline v1 (Manuelle) | Baseline v2 (Pipeline OOP) | Évolution            |
|--------------------------------------------------|------------------------|----------------------------|----------------------|
| **Score Privé** *(Private Score - 50% du test)*  | 0.8505                 | **0.8574**                 | **+0.0069 (+0.81%)** |
| **Score Public** *(Public Score - 50% du test)*  | **0.8564**<br>         | 0.8519                     | -0.0045 (-0.52%)     |
| **Moyenne Globale Test** *(Estimation sur 100%)* | 0.8534                 | **0.8546**                 | **+0.0012**          |
| **Score de Validation Locale** *(Cross-check)*   | 0.8570                 | **0.8580**                 | **+0.0010**          |

### Analyse des résultats de la soumission Kaggle v2

L'analyse de ces scores révèle un phénomène statistique particulièrement instructif sur la qualité de généralisation de notre modèle :

* **Une progression significative sur le jeu de test privé :** Dans les compétitions Kaggle, le score privé (*Private Score*) constitue le seul et unique critère de classement officiel, car il est calculé sur une portion du jeu de test totalement invisible pendant la phase de développement afin d'évaluer la réelle capacité de généralisation des modèles. Sur cet échantillon de référence, notre nouvelle baseline enregistre une hausse de **+0.0069 points d'AUC-ROC** par rapport au modèle initial (passant de 0.8505 à 0.8574). Ce gain prouve que la scission de la variable `DebtRatio` en une mensualité brute en euros (`MonthlyDebtAmount`) a apporté un signal prédictif universel, beaucoup plus robuste sur des populations neuves que l'ancien ratio hybride.


* **La réduction du sur-apprentissage (Effet Public vs Privé) :** Sur la première baseline, le modèle affichait un score public de 0.8564 pour un score privé de 0.8505, soit une chute de performance d'environ 0.006 points lorsque le modèle était confronté à la seconde moitié des données. Ce différentiel suggérait que les seuils de coupure codés en dur dans notre script initial (notamment le cap à 72 000 € sur les revenus) avaient légèrement sur-appris les particularités locales de notre jeu d'entraînement et du leaderboard public. Dans cette seconde version, l'écart entre le public (0.8519) et le privé (0.8574) s'est inversé à la hausse. L'utilisation du `QuantileCapper` et de l'algorithme `Kneed` a permis de régulariser la distribution en coupant les queues au niveau d'un vrai coude mathématique, rendant l'algorithme moins sensible au bruit et plus performant sur le jeu d'évaluation définitif.


* **La confirmation d'un environnement de validation local fiable :** Le score de notre validation locale sur `X_val` (0.8580) se trouve être extrêmement proche du score de test privé officiel obtenu sur la plateforme Kaggle (0.8574). Cette concordance quasi parfaite confirme l'étanchéité totale de notre pipeline orienté objet : l'application rigoureuse du `.fit_transform()` uniquement sur les données d'entraînement a éliminé tout risque de *Data Leakage*. Nous avons ainsi la garantie que notre stratégie de découpage local reflète fidèlement la réalité d'un environnement de production.

Cette deuxième baseline logistique, standardisée via Scikit-Learn et validée par ce score privé de **0.8574**, constitue dorénavant un référentiel rigoureux. Nous pouvons désormais aborder l'implémentation de notre modèle non linéaire XGBoost avec un objectif clair : vérifier si sa capacité à capturer des interactions complexes permet de surpasser ce seuil de performance sans dégrader la robustesse sur le jeu de test privé.

In [ ]:
from src.utils import save_pipeline

save_pipeline(baseline_pipeline, "logreg_baseline_v2.joblib")